### 1. Kütüphanelerin İçe Aktarılması ve Görsel Ayarlar
Gerekli veri bilimi, makine öğrenmesi ve görselleştirme kütüphanelerinin içe aktarılması. Ayrıca, grafiklerin tüm projede tutarlı görünmesi için görsel stillerin (seaborn stili, varsayılan boyutlar ve renk paleti) ayarlanması.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost
from xgboost import XGBClassifier

# Uyarıları gizle
warnings.filterwarnings('ignore')

# Görselleştirme stili ayarları
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Özel renk paleti: Pass, Fail, Distinction, Withdrawn
custom_palette = ['#0D9488', '#EF4444', '#6366F1', '#F97316']
sns.set_palette(custom_palette)

print("Kütüphaneler ve görsel ayarlar başarıyla yüklendi.")

### 2. Veri Setinin Yüklenmesi ve Hazırlanması
Temizlenmiş veri setinin (`dataset_clean.csv`) yüklenmesi, bağımsız değişkenler (X) ve bağımlı değişkenin (y) ayrılması. `target` ve `final_result` sütunları özelliklerden çıkarılacaktır.

In [ ]:
# Veri setini yükle
df = pd.read_csv('../data/processed/dataset_clean.csv')

# Hedef değişkeni (y) belirle
# Veri setinde 'target' veya 'final_result' değişkenlerinden birini y olarak alıyoruz.
if 'target' in df.columns:
    y = df['target']
elif 'final_result' in df.columns:
    y = df['final_result']
else:
    y = df.iloc[:, -1]

# X'i belirle (target ve final_result sütunlarını düşür)
cols_to_drop = [col for col in ['target', 'final_result'] if col in df.columns]
X = df.drop(columns=cols_to_drop)

print("Veri seti yüklendi ve X, y olarak ayrıldı.")
print(f"Tüm veri seti boyutu: {df.shape}")
print(f"X boyutu: {X.shape}")
print(f"y boyutu: {y.shape}")

### 3. Veri Bölme, SMOTE Uygulaması ve Modelin Eğitilmesi
Veri setinin eğitim ve test alt kümelerine bölünmesi (`test_size=0.2`, `stratify=y`). Sınıf dengesizliğini gidermek için yalnızca eğitim setine SMOTE uygulanması. Ardından en iyi performansı gösteren Random Forest modelinin yeniden eğitilmesi.

In [ ]:
# Train/test split (%80 eğitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Sadece eğitim setine SMOTE uygula
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Random Forest modelini belirlenen parametrelerle yeniden eğit
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    class_weight='balanced', 
    n_jobs=-1
)
rf_model.fit(X_train_resampled, y_train_resampled)

print("Eğitim ve test setleri bölündü. Eğitim setine SMOTE uygulandı ve Random Forest eğitildi.")
print(f"Eğitim Seti Şekli (SMOTE Sonrası): X={X_train_resampled.shape}, y={y_train_resampled.shape}")
print(f"Test Seti Şekli: X={X_test.shape}, y={y_test.shape}")

### 4. Özellik Önem Derecelerinin Çıkarılması ve Görselleştirilmesi
Random Forest modelinden her bir özelliğin modelin karar sürecindeki önem dereceleri çıkarılır. En önemli 20 özellik belirlenerek, önem derecelerine göre renklendirilmiş (gradient) yatay bir çubuk grafik oluşturulur.

In [ ]:
import os

# Görsellerin kaydedileceği klasörü oluştur
os.makedirs('../visuals/eda', exist_ok=True)

# 1. Özellik önem derecelerini çıkar
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Önem derecesine göre azalan şekilde sırala
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# En önemli 20 özelliği al
top_20_features = feature_importance_df.head(20)

# 2. En önemli 20 özelliği yatay çubuk grafik ile çizdir
plt.figure(figsize=(12, 8))

# Gradient renk paleti (koyudan açığa)
colors = sns.color_palette("Blues_r", n_colors=20) 

ax = sns.barplot(x='Importance', y='Feature', data=top_20_features, palette=colors)

# Barların sonuna değerleri ekle
for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}", 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', 
                xytext=(5, 0), 
                textcoords='offset points',
                fontsize=10)

plt.title("En Önemli 20 Özellik (Random Forest)", fontsize=16)
plt.xlabel("Önem Derecesi", fontsize=12)
plt.ylabel("Özellik Adı", fontsize=12)
plt.tight_layout()

# Grafiği kaydet ve göster
plt.savefig('../visuals/eda/feature_importance_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

### 5. Özellik Gruplarının Katkı Payı (Pie Chart)
Özellikler kategorik gruplara ayrılarak (VLE etkileşimleri, Değerlendirme notları, Demografik veriler) her grubun toplam modele katkısı pasta grafiği ile gösterilir. Bu sayede öğrencilerin başarı durumunu etkileyen ana faktörlerin hangi alanlarda yoğunlaştığı analiz edilebilir.

In [ ]:
# 3. Özellikleri kategorilerine göre gruplandır
vle_features = ['sum_clicks', 'active_days', 'early_clicks', 'first_interaction', 'last_interaction']
assessment_features = ['avg_score', 'weighted_avg_score', 'submission_count', 'late_submission_count', 'late_submission_rate']

def categorize_feature(feature_name):
    if any(vle in feature_name for vle in vle_features):
        return 'VLE (Etkileşim) Özellikleri'
    elif any(ass in feature_name for ass in assessment_features):
        return 'Değerlendirme (Not) Özellikleri'
    else:
        return 'Demografik ve Diğer Özellikler'

feature_importance_df['Group'] = feature_importance_df['Feature'].apply(categorize_feature)

# Gruplara göre toplam önem derecesini hesapla
group_importance = feature_importance_df.groupby('Group')['Importance'].sum().sort_values(ascending=False)

# Pasta grafiği ile görselleştir
plt.figure(figsize=(8, 8))
colors = ['#6366F1', '#0D9488', '#F97316'] # Sıraya göre renkler
plt.pie(group_importance, labels=group_importance.index, autopct='%1.1f%%', startangle=140, colors=colors)
plt.title("Özellik Gruplarının Modele Toplam Katkısı", fontsize=14)
plt.tight_layout()

# Grafiği kaydet ve göster
plt.savefig('../visuals/eda/feature_group_contribution.png', dpi=300)
plt.show()

# Sonuçları ekrana yazdır
print("Grup Bazında Toplam Önem Dereceleri:")
print(group_importance)

### Analiz ve Yorum
Yukarıdaki pasta grafiğinde özellik gruplarının katkıları incelendiğinde, genellikle **VLE (Etkileşim) Özellikleri** veya **Değerlendirme (Not) Özellikleri** grubunun en büyük katkıyı sağladığı görülmektedir. Öğrencilerin platformla olan etkileşimleri (`sum_clicks`, `active_days`) ve sınavlardaki başarıları (`avg_score`), onların nihai başarı durumunu tahmin etmede en belirleyici faktörlerdir. Demografik veriler ise tahmine daha düşük oranda ancak destekleyici bir katkı sunmaktadır.

### 6. Öğrenme Eğrisi (Learning Curve) Analizi
Modelin eğitim ve çapraz doğrulama (cross-validation) performansının eğitim seti boyutu arttıkça nasıl değiştiğini gözlemlemek için öğrenme eğrisi çizdirilir. Bu analiz modelin overfit (aşırı öğrenme) veya underfit (eksik öğrenme) durumunu değerlendirmek için önemlidir.

In [ ]:
from sklearn.model_selection import learning_curve

# 4. Öğrenme eğrisi fonksiyonu ile skorların hesaplanması
# Uyarı: Bu işlem veri setinin büyüklüğüne göre biraz zaman alabilir
train_sizes, train_scores, test_scores = learning_curve(
    estimator=rf_model, 
    X=X_train_resampled, 
    y=y_train_resampled, 
    train_sizes=np.linspace(0.1, 1.0, 10), 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1,
    random_state=42
)

# Ortalama ve standart sapmaları hesapla
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Grafiği çiz
plt.figure(figsize=(10, 6))

plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1, color="#0D9488")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color="#EF4444")

plt.plot(train_sizes, train_scores_mean, 'o-', color="#0D9488", label="Eğitim Skoru")
plt.plot(train_sizes, test_scores_mean, 'o-', color="#EF4444", label="Çapraz Doğrulama (Validation) Skoru")

plt.title("Öğrenme Eğrisi (Random Forest)", fontsize=16)
plt.xlabel("Eğitim Seti Boyutu", fontsize=12)
plt.ylabel("F1 Macro Skoru", fontsize=12)
plt.legend(loc="best")
plt.grid(True)
plt.tight_layout()

# Grafiği kaydet ve göster
plt.savefig('../visuals/eda/learning_curve.png', dpi=300)
plt.show()

### Öğrenme Eğrisi Yorumu
* **Aşırı Öğrenme (Overfitting) / Eksik Öğrenme (Underfitting) Durumu:** Eğer eğitim skoru çok yüksek (1.0'a yakın) ancak doğrulama (validation) skoru düşük kalıyorsa modelde aşırı öğrenme (overfitting) vardır. Eğer iki skor da düşükse eksik öğrenme (underfitting) yaşanmaktadır.
* **Yakınsama (Convergence):** Eğitim seti boyutu arttıkça eğitim skoru yavaşça düşerken, doğrulama skorunun artarak birbirine yaklaşması (yakınsaması) ideal bir modelin işaretidir. Eğer eğriler birbirine yaklaşıyorsa, model daha fazla veriden faydalanıyor demektir; ancak arada hala geniş bir boşluk varsa (gap), modelin hiperparametrelerini optimize etmek veya regularization teknikleri kullanmak gerekebilir.

### 7. Tahmin ve Gerçek Değerlerin Karşılaştırılması
Test seti üzerinden tahminler alınarak gerçek (`y_test`) ve tahmin edilen (`y_pred`) değerler karşılaştırılır. Bu sayede modelin genel sınıf dağılımında hangi sınıfları ne oranda doğru veya yanlış tahmin ettiği görülür.

In [ ]:
# 5. Tahminleri al ve bir DataFrame oluştur
y_pred = rf_model.predict(X_test)

# Sınıf isimleri (Orijinal veri setindeki değerler olduğunu varsayıyoruz)
class_names = ['Pass', 'Fail', 'Distinction', 'Withdrawn']

# Sonuçları DataFrame'e dönüştür
results_df = pd.DataFrame({
    'Gerçek': y_test,
    'Tahmin': y_pred
})
results_df['Doğru_mu'] = results_df['Gerçek'] == results_df['Tahmin']

# Dağılımları hesapla
actual_counts = results_df['Gerçek'].value_counts().reindex(class_names).fillna(0)
pred_counts = results_df['Tahmin'].value_counts().reindex(class_names).fillna(0)

# Gruplanmış çubuk grafik için veri hazırlığı
x_indices = np.arange(len(class_names))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x_indices - width/2, actual_counts, width, label='Gerçek Sınıf', color='#6366F1')
plt.bar(x_indices + width/2, pred_counts, width, label='Tahmin Edilen Sınıf', color='#F97316')

plt.title("Gerçek vs Tahmin Edilen Sınıf Dağılımı", fontsize=16)
plt.xlabel("Sınıflar", fontsize=12)
plt.ylabel("Öğrenci Sayısı", fontsize=12)
plt.xticks(x_indices, class_names)
plt.legend()
plt.tight_layout()

# Grafiği kaydet
plt.savefig('../visuals/eda/pred_vs_actual.png', dpi=300)
plt.show()

print(f"Toplam Test Verisi: {len(results_df)}")
print(f"Doğru Tahmin Sayısı: {results_df['Doğru_mu'].sum()} ({(results_df['Doğru_mu'].sum() / len(results_df)) * 100:.2f}%)")
print(f"Yanlış Tahmin Sayısı: {len(results_df) - results_df['Doğru_mu'].sum()}")

### 8. Hata Analizi (Hangi Sınıf Neyle Karıştı?)
Modelin yanıldığı (yanlış tahmin edilen) durumlara odaklanarak, gerçekte bir sınıfa ait olan öğrencilerin ağırlıklı olarak hangi başka sınıfa dahil edildiği analiz edilir. Bu görsel, bir hata matrisi (confusion matrix) mantığıyla oluşturulur ancak sadece hatalı tahminlere odaklanır.

In [ ]:
# 6. Yalnızca yanlış tahmin edilen verileri filtrele
errors_df = results_df[~results_df['Doğru_mu']]

# Hata matrisini oluştur
error_matrix = pd.crosstab(errors_df['Gerçek'], errors_df['Tahmin'], 
                           rownames=['Gerçek Sınıf'], colnames=['Tahmin Edilen Sınıf'])

# Tüm sınıfların matriste yer almasını sağla (hata olmayan sınıflar için)
error_matrix = error_matrix.reindex(index=class_names, columns=class_names, fill_value=0)

# Köşegenleri (doğru tahminleri) sıfırla ki sadece karışanlar görünsün (zaten filtrelenmiş olsa da güvenlik için)
for cls in class_names:
    error_matrix.loc[cls, cls] = 0

# Satır bazında normalize et (Gerçekte X olanların hata yapılan kısmının yüzde kaçı Y'ye karışmış?)
error_matrix_norm = error_matrix.div(error_matrix.sum(axis=1), axis=0).fillna(0) * 100

# Heatmap ile görselleştir
plt.figure(figsize=(10, 8))
sns.heatmap(error_matrix_norm, annot=True, fmt=".1f", cmap="Reds", cbar_kws={'label': 'Hata Yüzdesi (%)'})
plt.title("Hata Analizi: Hangi Sınıf Neyle Karıştı?", fontsize=16)
plt.tight_layout()

# Grafiği kaydet
plt.savefig('../visuals/eda/error_analysis_heatmap.png', dpi=300)
plt.show()

### Hata Analizi Yorumu
Yukarıdaki ısı haritasında (heatmap), modelin en çok hangi iki sınıf arasında kararsız kaldığı ve hata yaptığı görülmektedir.
* **En Çok Karışan Sınıflar:** Genellikle `Fail` (Başarısız) ve `Withdrawn` (Dersten Çekilen) sınıfları birbirine veya `Pass` (Geçen) sınıfıyla karıştırılır. Bunun nedeni etkileşimi düşük öğrencilerin hem başarısız olma hem de dersi bırakma eğiliminde olmalarıdır.
* **Gerçek Dünya Etkisi:** `Withdrawn` (Dersten Çekilen) öğrencilerin yanlışlıkla `Pass` veya `Distinction` olarak tahmin edilmesi veya tersi büyük bir risktir. Eğitmenlerin sistemi terk etme eğilimindeki öğrencileri önceden tespit edememesi (**erken müdahale - early intervention eksikliği**), öğrenci tutma oranlarını (retention rate) ciddi şekilde düşürebilir. Eğitim kurumları için öğrencilerin sadece geçip kalması değil, dersi tamamen bırakması çok daha kritik bir veri olduğundan, `Withdrawn` sınıfındaki hataları minimize edecek stratejiler (eşik düşürme veya ağırlıklandırma) uygulanmalıdır.

### 9. Model Özet Panosu (Dashboard)
Tüm önemli analizleri (özellik önemi, hata matrisi, tahmin dağılımı ve özellik grup katkısı) tek bir görsel üzerinde toplayarak yöneticiler ve eğitmenler için kapsamlı bir 'Özet Panosu' (Dashboard) oluşturulur.

In [ ]:
import os

# 2x2 alt grafik düzeni oluştur
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- Sol Üst: En Önemli 10 Özellik ---
top_10 = feature_importance_df.head(10)
colors_10 = sns.color_palette("Blues_r", n_colors=10)
sns.barplot(x='Importance', y='Feature', data=top_10, palette=colors_10, ax=axes[0, 0])
axes[0, 0].set_title("En Önemli 10 Özellik", fontsize=14)
axes[0, 0].set_xlabel("Önem Derecesi")
axes[0, 0].set_ylabel("")

# --- Sağ Üst: Karmaşıklık Matrisi (Normalize Edilmiş) ---
cm = confusion_matrix(y_test, y_pred, labels=class_names)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0, 1])
axes[0, 1].set_title("Karmaşıklık Matrisi (Doğruluk %)", fontsize=14)
axes[0, 1].set_xlabel("Tahmin Edilen")
axes[0, 1].set_ylabel("Gerçek Sınıf")

# --- Sol Alt: Gerçek vs Tahmin Edilen Dağılımı ---
axes[1, 0].bar(x_indices - width/2, actual_counts, width, label='Gerçek', color='#6366F1')
axes[1, 0].bar(x_indices + width/2, pred_counts, width, label='Tahmin', color='#F97316')
axes[1, 0].set_title("Sınıf Dağılımı Karşılaştırması", fontsize=14)
axes[1, 0].set_xticks(x_indices)
axes[1, 0].set_xticklabels(class_names)
axes[1, 0].set_ylabel("Öğrenci Sayısı")
axes[1, 0].legend()

# --- Sağ Alt: Özellik Gruplarının Katkısı ---
colors_pie = ['#6366F1', '#0D9488', '#F97316']
axes[1, 1].pie(group_importance, labels=group_importance.index, autopct='%1.1f%%', 
               startangle=140, colors=colors_pie)
axes[1, 1].set_title("Özellik Gruplarının Katkısı", fontsize=14)

# --- Ana Başlık ve Düzen ---
fig.suptitle("Öğrenci Başarı Tahmini — Model Özet Panosu (Random Forest)", fontsize=20, y=1.02, fontweight='bold')
plt.tight_layout()

# Grafiği kaydet ve göster
plt.savefig('../visuals/eda/model_ozet_panosu.png', dpi=300, bbox_inches='tight')
plt.show()

### 10. Sonuç ve Sonraki Adımlar
Oluşturulan tüm görseller `visuals/eda/` klasörüne kaydedilmiştir. Aşağıdaki kod ile üretilen dosyalar listelenmektedir.

In [ ]:
# Kaydedilen görselleri listele
print("Kaydedilen Görseller (visuals/eda/):")
for file in os.listdir('../visuals/eda/'):
    if file.endswith('.png'):
        print(f" - {file}")

print("\nGörselleştirme tamamlandı. Sıradaki adım:\nFinal Raporu (feature/final-report)")